In [ ]:
# --------------------------
# Install and import libraries
# --------------------------
# pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# --------------------------
# Load dataset
# --------------------------
df = pd.read_csv("malicious.csv")  
df.head()  #"sneak in and look a glance at dataset

# --------------------------
# Dataset checking
# --------------------------
df.info()
df.isnull().sum()
df.sample(5)
df.sample(10)
df.info()

# --------------------------
# Encode output labels
# --------------------------
# --------------------------
# Encode output labels and save mapping
# --------------------------
from sklearn.preprocessing import LabelEncoder
import pickle

le = LabelEncoder()
df['label'] = le.fit_transform(df['type'])

# Save label mapping for UI
label_map = dict(zip(le.transform(le.classes_), le.classes_))
with open('label_map.pkl', 'wb') as f:
    pickle.dump(label_map, f)

df.head()
list(zip(le.classes_, le.transform(le.classes_)))



# --------------------------
# Data cleaning
# --------------------------
df.dropna(subset=['url'], inplace=True)
df.drop_duplicates(subset=['url'], inplace=True)

df['url'] = df['url'].str.lower()
df['url'] = df['url'].str.replace(r'https?://', '', regex=True)
df['url'] = df['url'].str.replace(r'www\.', '', regex=True)
df['url'] = df['url'].str.strip()
df.sample(5)

# --------------------------
# Sample 10k URLs for training
# --------------------------
# --------------------------
# Sample 40k URLs for training
# --------------------------
X_sample = df['url']
y_sample = df['label']

# --------------------------
# TF-IDF vectorization
# --------------------------
vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3,4),
    max_features=3000   # reduce for RAM
)

X_tfidf = vectorizer.fit_transform(X_sample)
print("TF-IDF shape:", X_tfidf.shape)

# --------------------------
# Train-test split
# --------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_sample, test_size=0.2, random_state=42, stratify=y_sample
)

# --------------------------
# Train Logistic Regression
# --------------------------
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

model = OneVsRestClassifier(LogisticRegression(max_iter=300))
model.fit(X_train, y_train)

# --------------------------
# Test model
# --------------------------
y_pred = model.predict(X_test)
print(y_pred)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# --------------------------
# Save model & vectorizer
# --------------------------
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     651191 non-null  object
 1   type    651191 non-null  object
dtypes: object(2)
memory usage: 9.9+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     651191 non-null  object
 1   type    651191 non-null  object
dtypes: object(2)
memory usage: 9.9+ MB


In [ ]:
import tkinter as tk
from tkinter import messagebox
import pickle
import re

# --------------------------
# Load model, vectorizer & label map
# --------------------------
with open('vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('label_map.pkl', 'rb') as f:
    label_map = pickle.load(f)

# --------------------------
# Define URL check function
# --------------------------
def check_url():
    url = entry.get().strip().lower()

    if not url:
        messagebox.showwarning("Empty URL", "Please enter a URL!")
        return

    url = re.sub(r'https?://', '', url)
    url = re.sub(r'www\.', '', url)

    X_new = vectorizer.transform([url])
    pred = model.predict(X_new)[0]

    result = label_map[pred]

    if result.lower() in ("benign", "safe"):
        color = "#2ecc71"   # green
        emoji = "🟢"
    else:
        color = "#e74c3c"   # red
        emoji = "🔴"

    result_label.config(text=f"{emoji} Prediction: {result}", fg=color)


# --------------------------
# Tkinter UI Setup
# --------------------------
root = tk.Tk()
root.title("🚨 URL Malicious Detector")
root.geometry("540x260")

# Full dark background color (deep navy/yellowish-black)
BG_COLOR = "#0b0f1a"   # dark blue/black
root.config(bg=BG_COLOR)

# Title Label
title = tk.Label(
    root,
    text="🔍 URL Safety Checker",
    font=("Helvetica", 20, "bold"),
    fg="#ffffff",
    bg=BG_COLOR
)
title.pack(pady=10)

# Instruction Label
tk.Label(
    root,
    text="Enter a URL to test if it's safe or malicious:",
    font=("Arial", 12),
    fg="#c7c7c7",
    bg=BG_COLOR
).pack(pady=5)

# Entry box
entry = tk.Entry(
    root,
    width=50,
    font=("Arial", 13),
    bd=2,
    relief="solid",
    bg="#1a1f2e",
    fg="#ffffff",
    insertbackground="#ffffff"
)
entry.pack(pady=10)

# Check button
check_btn = tk.Button(
    root,
    text="Check URL 🚀",
    font=("Arial", 12, "bold"),
    bg="#4834d4",
    fg="white",
    activebackground="#686de0",
    activeforeground="white",
    width=15,
    cursor="hand2",
    command=check_url,
    bd=0
)
check_btn.pack(pady=10)

# Result Label
result_label = tk.Label(
    root,
    text="",
    font=("Arial", 14, "bold"),
    bg=BG_COLOR
)
result_label.pack(pady=10)

# Run app
root.mainloop()
